# Taller 1. Formulación y resolución de un modelo en red

**Investigación de Operaciones**
Doctorado en Ingeniería, consorcio UV–UTA
Jueves 10 de septiembre de 2026, Bloque B

---

Integrantes:Pedro Limarí, Diego Baltazar
Instancia asignada:Industria 4.0
Fecha:septiembre 2026

---

Esta plantilla entrega la estructura común a las cuatro instancias. Lo que falta —y es lo que se
evalúa— son la formulación, la interpretación y la verificación. Los bloques marcados con
`# TODO` deben completarse.

Regla dura del taller: **ningún dato numérico se escribe dentro del modelo**. Todo se lee de los
archivos CSV de la carpeta `datos/`.


## Paso 0. Verificación del entorno


In [25]:
import sys
print("Python", sys.version.split()[0])

import pyomo.environ as pyo
print("Pyomo", pyo.__version__ if hasattr(pyo, "__version__") else "instalado")

solver = pyo.SolverFactory("appsi_highs")
print("HiGHS disponible:", solver.available())

import pandas as pd
print("pandas", pd.__version__)


Python 3.13.15
Pyomo instalado
HiGHS disponible: True
pandas 2.2.3


## Paso 1. Lectura de los datos

Ajuste `CARPETA` a la instancia que le fue asignada. El separador de los CSV es el punto y coma.


In [26]:
from pathlib import Path
import pandas as pd

CARPETA = Path("datos")

tiempos = pd.read_csv(CARPETA / "tiempos.csv", sep=";")
restriccion = pd.read_csv(CARPETA / "restriccion_adicional.csv", sep=";")

display(tiempos)
display(restriccion)

,orden,C1,C2,C3,C4,C5,C6
0,O1,42,55,61,38,70,49
1,O2,58,40,44,63,52,47
2,O3,36,62,39,51,58,66
3,O4,49,43,57,45,41,53
4,O5,64,51,46,59,37,42
5,O6,45,60,50,44,55,38


,descripcion,celdas,tope_minutos
0,Las celdas C1 y C2 comparten un unico brazo de...,C1+C2,77


**Interpretación de los datos:** `tiempos.csv` contiene los tiempos, en minutos, asociados a cada combinación orden-celda. `restriccion_adicional.csv` identifica las celdas que comparten el brazo de carga y el límite de tiempo que se incorporará posteriormente al modelo.

## Paso 2. Estructura de la red

Antes de programar nada, escriba en palabras qué representa cada nodo y cada arco, y con qué
unidades. Un modelo cuyas unidades no cierran está mal aunque el solver entregue un número.


### Interpretación de la red

La instancia se representa mediante una red bipartita dirigida, formada por dos conjuntos de nodos:

- **Nodos de órdenes:** $O_1,\ldots,O_6$. Cada orden aporta una unidad de flujo al sistema, por lo que, bajo la convención

$$
\text{salidas}-\text{entradas}=q_i,
$$

se tiene $q_i=+1$ para cada nodo de orden.

- **Nodos de celdas robotizadas:** $C_1,\ldots,C_6$. Cada celda debe recibir exactamente una orden, por lo que su flujo exógeno es $q_j=-1$.

Un arco dirigido $(O_i,C_j)$ representa la posibilidad de asignar la orden $i$ a la celda $j$.

El parámetro $t_{ij}$ asociado al arco representa el tiempo requerido para procesar la orden $i$ en la celda $j$, expresado en **minutos por unidad de orden asignada**.

La variable $x_{ij}$ representa el flujo de asignación sobre el arco $(O_i,C_j)$ y se expresa en unidades de orden, con:

$$
0\leq x_{ij}\leq1
$$

Por lo tanto, dimensionalmente:

$$
\frac{\text{minutos}}{\text{orden}}
\times
\text{orden}
=
\text{minutos}
$$

de modo que $t_{ij}x_{ij}$ representa el tiempo aportado por esa asignación a la función objetivo.

Como existen seis órdenes y seis celdas, la red está balanceada:

$$
\sum_{n\in N}q_n
=
6(+1)+6(-1)
=
0
$$

In [27]:
# Órdenes y celdas obtenidas desde el CSV
ordenes = tiempos["orden"].tolist()
celdas = [col for col in tiempos.columns if col != "orden"]

# Conjunto de nodos
N = ordenes + celdas

# Flujo exógeno:
# +1 para cada orden, -1 para cada celda
q = {o: 1 for o in ordenes}
q.update({c: -1 for c in celdas})

# Arcos: (orden, celda) -> (costo, cota inferior, cota superior)
A = {}

for _, fila in tiempos.iterrows():
    orden = fila["orden"]

    for celda in celdas:
        A[(orden, celda)] = (fila[celda], 0, 1)

# Comprobación
print("suma de los flujos exógenos:", sum(q.values()))

suma de los flujos exógenos: 0


## Paso 3. El modelo

$$\min \; \sum_{(i,j)\in A} c_{ij}\,x_{ij}
\quad\text{s.a.}\quad
\sum_{j} x_{ij} - \sum_{k} x_{ki} = q_i \;\; \forall i \in N,
\qquad l_{ij} \le x_{ij} \le u_{ij}$$

Note que las variables se declaran **continuas**. No se impone integralidad: si el modelo es
realmente de red y el lado derecho es entero, la solución saldrá entera por sí sola. Comprobarlo es
parte del taller.


### Formulación del modelo para la Instancia D – Industria 4.0

Se definen los conjuntos:

$$
O=\{O_1,\ldots,O_6\}
$$

de órdenes, y

$$
C=\{C_1,\ldots,C_6\}
$$

de celdas robotizadas. El conjunto total de nodos es:

$$
N=O\cup C
$$

y, dado que cualquier orden puede ser asignada a cualquier celda, el conjunto de arcos es:

$$
A=O\times C
$$

Para cada arco $(i,j)\in A$, el parámetro $t_{ij}$ representa el tiempo requerido para procesar la orden $i$ en la celda $j$, expresado en minutos por unidad de orden asignada.

El flujo exógeno de cada nodo se define como:

$$
q_n=
\begin{cases}
+1, & n\in O,\\
-1, & n\in C.
\end{cases}
$$

La variable de decisión $x_{ij}$ representa el flujo de asignación sobre el arco $(i,j)$. En esta primera etapa se declara continua:

$$
0\leq x_{ij}\leq1
\qquad \forall (i,j)\in A
$$

La función objetivo minimiza el tiempo total de procesamiento:

$$
\min z=
\sum_{(i,j)\in A}t_{ij}x_{ij}
$$

Cada orden debe asignarse exactamente a una celda:

$$
\sum_{j\in C}x_{ij}=1
\qquad \forall i\in O
$$

y cada celda debe recibir exactamente una orden:

$$
\sum_{i\in O}x_{ij}=1
\qquad \forall j\in C
$$

Equivalentemente, utilizando la formulación de flujo a costo mínimo, estas restricciones pueden escribirse mediante conservación de flujo:

$$
\sum_{j:(n,j)\in A}x_{nj}
-
\sum_{i:(i,n)\in A}x_{in}
=
q_n
\qquad \forall n\in N
$$

En esta etapa **no se impone integralidad**. El propósito es verificar posteriormente si la estructura de red y el lado derecho entero son suficientes para que la relajación lineal produzca naturalmente una solución 0-1.

In [28]:
import pyomo.environ as pyo

m = pyo.ConcreteModel()
m.N = pyo.Set(initialize=list(N))
m.A = pyo.Set(initialize=list(A), dimen=2)

m.x = pyo.Var(m.A, domain=pyo.NonNegativeReals,
              bounds=lambda m, i, j: (A[(i, j)][1], A[(i, j)][2]))

m.obj = pyo.Objective(expr=sum(A[a][0] * m.x[a] for a in m.A), sense=pyo.minimize)

def balance(m, n):
    sale  = sum(m.x[i, j] for (i, j) in m.A if i == n)
    entra = sum(m.x[i, j] for (i, j) in m.A if j == n)

    # En la Instancia D se usa igualdad en todos los nodos,
    # porque cada orden debe asignarse exactamente una vez
    # y cada celda debe recibir exactamente una orden.
    return sale - entra == q[n]

m.bal = pyo.Constraint(m.N, rule=balance)

# Los valores duales deben declararse ANTES de resolver, o no se importan.
m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

print("variables:", len(m.A), " restricciones:", len(m.N))


variables: 36  restricciones: 12


## Paso 4. Resolución del Modelo Base


### Resolución del modelo base

Se resuelve el problema de asignación original utilizando el solver HiGHS.

En esta etapa las variables $x_{ij}$ continúan siendo **continuas entre 0 y 1**. No se ha incorporado todavía la restricción adicional asociada al brazo de carga compartido por C1 y C2.

El objetivo es determinar la asignación orden-celda que minimiza el tiempo total de procesamiento.

In [29]:
res = pyo.SolverFactory("appsi_highs").solve(m)
print("condición de término:", res.solver.termination_condition)
print("valor óptimo: ", pyo.value(m.obj))

flujos = {a: pyo.value(m.x[a]) for a in m.A}
for a, v in sorted(flujos.items()):
    if v > 1e-6:
        print(f"   {a[0]:>10} -> {a[1]:<10} {v:>12,.2f}   de {A[a][2]:,.0f} de capacidad")


condición de término: optimal
valor óptimo:  236.0
           O1 -> C4                 1.00   de 1 de capacidad
           O2 -> C3                 1.00   de 1 de capacidad
           O3 -> C1                 1.00   de 1 de capacidad
           O4 -> C2                 1.00   de 1 de capacidad
           O5 -> C5                 1.00   de 1 de capacidad
           O6 -> C6                 1.00   de 1 de capacidad


### Interpretación de la solución

El solver HiGHS encontró una solución óptima para el modelo base con un tiempo total de:

$$
z^*=236\text{ minutos}
$$

La asignación óptima obtenida es:

- $O_1\rightarrow C_4$: 38 minutos
- $O_2\rightarrow C_3$: 44 minutos
- $O_3\rightarrow C_1$: 36 minutos
- $O_4\rightarrow C_2$: 43 minutos
- $O_5\rightarrow C_5$: 37 minutos
- $O_6\rightarrow C_6$: 38 minutos

La suma de los tiempos asociados a las seis asignaciones es:

$$
38+44+36+43+37+38=236
$$

Por lo tanto, el tiempo mínimo total del problema de asignación base es de **236 minutos**.

Aunque las variables fueron declaradas continuas en el intervalo $[0,1]$, la solución obtenida presenta únicamente valores 0 y 1. En este punto se constata empíricamente dicha integralidad; su justificación teórica se analizará posteriormente mediante la estructura de la matriz de incidencia.

## Paso 5. Verificación de la conservación de flujo

Esta comprobación es obligatoria y vale puntaje. No basta con afirmar que el solver la respetó.


### Verificación de la conservación de flujo

Una vez obtenida la solución óptima, se verifica numéricamente la ecuación de conservación de flujo en cada nodo:

$$
\sum_j x_{ij}-\sum_k x_{ki}=q_i
\qquad \forall i\in N
$$

Bajo la convención utilizada, los nodos de órdenes tienen $q_i=+1$, por lo que debe salir exactamente una unidad de flujo, mientras que los nodos de celdas tienen $q_j=-1$, por lo que debe entrar exactamente una unidad.

La verificación se realiza **nodo por nodo**, comparando el flujo neto calculado con el flujo exógeno exigido. De esta forma se comprueba directamente que cada orden fue asignada exactamente una vez y que cada celda recibió exactamente una orden.

In [30]:
print(f"{'nodo':<10}{'sale':>12}{'entra':>12}{'neto':>12}{'exigido':>12}   estado")
todo_ok = True
for n in N:
    sale  = sum(v for (i, j), v in flujos.items() if i == n)
    entra = sum(v for (i, j), v in flujos.items() if j == n)
    neto  = sale - entra
    ok = abs(neto - q[n]) < 1e-6
    todo_ok &= ok
    print(f"{n:<10}{sale:>12,.2f}{entra:>12,.2f}{neto:>12,.2f}{q[n]:>12,.2f}   {'ok' if ok else 'ERROR'}")
print("\nconservación de flujo verificada en todos los nodos:", todo_ok)


nodo              sale       entra        neto     exigido   estado
O1                1.00        0.00        1.00        1.00   ok
O2                1.00        0.00        1.00        1.00   ok
O3                1.00        0.00        1.00        1.00   ok
O4                1.00        0.00        1.00        1.00   ok
O5                1.00        0.00        1.00        1.00   ok
O6                1.00        0.00        1.00        1.00   ok
C1                0.00        1.00       -1.00       -1.00   ok
C2                0.00        1.00       -1.00       -1.00   ok
C3                0.00        1.00       -1.00       -1.00   ok
C4                0.00        1.00       -1.00       -1.00   ok
C5                0.00        1.00       -1.00       -1.00   ok
C6                0.00        1.00       -1.00       -1.00   ok

conservación de flujo verificada en todos los nodos: True


### Interpretación de la verificación

La conservación de flujo se verifica correctamente en los 12 nodos de la red.

Para cada nodo de orden se cumple:

$$
1-0=+1
$$

lo que significa que cada orden entrega exactamente una unidad de asignación.

Para cada nodo correspondiente a una celda se cumple:

$$
0-1=-1
$$

lo que significa que cada celda recibe exactamente una orden.

La solución óptima de **236 minutos** satisface numéricamente las 12 ecuaciones de balance y, por tanto, las restricciones de asignación asociadas a la red.

## Paso 6. Integralidad

¿Salieron enteros los flujos? ¿Por qué? La respuesta debe nombrar la propiedad de la matriz y la
condición sobre el lado derecho, no limitarse a constatar el hecho.


### ¿Salieron enteros los flujos? ¿Por qué?

Sí. Aunque las variables $x_{ij}$ fueron declaradas continuas entre 0 y 1, la solución óptima obtenida fue 0-1.

La razón es que la matriz de restricciones asociada a la red es una **matriz totalmente unimodular** y, además, el vector del lado derecho es entero:

$$
q_n\in\{-1,+1\}
$$

para todos los nodos.

Bajo estas condiciones, los vértices de la región factible del modelo lineal son enteros. Por eso, aun permitiendo valores fraccionarios entre 0 y 1, el solver puede encontrar una solución óptima entera sin necesidad de declarar las variables como binarias.

In [31]:
no_enteros = {a: v for a, v in flujos.items() if abs(v - round(v)) > 1e-6}
print("flujos no enteros:", len(no_enteros))
if no_enteros:
    print(no_enteros)

# TODO: construya la matriz de incidencia nodo-arco y calcule el determinante de al menos
#       tres submatrices cuadradas. Verifique que todos caen en {0, 1, -1}.


flujos no enteros: 0


In [32]:
import numpy as np

# Matriz de incidencia nodo-arco
B = pd.DataFrame(
    0,
    index=N,
    columns=pd.MultiIndex.from_tuples(
        list(A.keys()),
        names=["origen", "destino"]
    )
)

# +1 en el nodo de origen y -1 en el nodo de destino
for i, j in A:
    B.loc[i, (i, j)] = 1
    B.loc[j, (i, j)] = -1

print("Dimensión de la matriz de incidencia:", B.shape)
display(B)

Dimensión de la matriz de incidencia: (12, 36)


origen  O1                O2           ... O5          O6               
destino C1 C2 C3 C4 C5 C6 C1 C2 C3 C4  ... C3 C4 C5 C6 C1 C2 C3 C4 C5 C6
O1       1  1  1  1  1  1  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
O2       0  0  0  0  0  0  1  1  1  1  ...  0  0  0  0  0  0  0  0  0  0
O3       0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
O4       0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
O5       0  0  0  0  0  0  0  0  0  0  ...  1  1  1  1  0  0  0  0  0  0
O6       0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  1  1  1  1  1  1
C1      -1  0  0  0  0  0 -1  0  0  0  ...  0  0  0  0 -1  0  0  0  0  0
C2       0 -1  0  0  0  0  0 -1  0  0  ...  0  0  0  0  0 -1  0  0  0  0
C3       0  0 -1  0  0  0  0  0 -1  0  ... -1  0  0  0  0  0 -1  0  0  0
C4       0  0  0 -1  0  0  0  0  0 -1  ...  0 -1  0  0  0  0  0 -1  0  0
C5       0  0  0  0 -1  0  0  0  0  0  ...  0  0 -1  0  0  0  0  0 -1  0
C6       0  0  0  0  0 -1  0  0  0  0  ...  0  0  0 -1  0  0  0  0  0 -1

[12 rows x 36 columns]

In [33]:
# Tres submatrices cuadradas para ilustrar
# la total unimodularidad

S1 = B.loc[
    ["O1", "C1"],
    [("O1", "C1"), ("O1", "C2")]
]

S2 = B.loc[
    ["O1", "C1"],
    [("O1", "C1"), ("O2", "C1")]
]

S3 = B.loc[
    ["O1", "O2"],
    [("O1", "C1"), ("O1", "C2")]
]

submatrices = {
    "Submatriz 1": S1,
    "Submatriz 2": S2,
    "Submatriz 3": S3
}

for nombre, S in submatrices.items():
    det = round(np.linalg.det(S.to_numpy(dtype=float)))

    print("\n", nombre)
    display(S)
    print("Determinante =", det)


 Submatriz 1


origen  O1   
destino C1 C2
O1       1  1
C1      -1  0

Determinante = 1

 Submatriz 2


origen,O1,O2
destino,C1,C1
O1,1,0
C1,-1,-1


Determinante = -1

 Submatriz 3


origen  O1   
destino C1 C2
O1       1  1
O2       0  0

Determinante = 0


### Interpretación

La solución base no presenta variables fraccionarias: todos los flujos obtenidos son 0 o 1.

Las tres submatrices analizadas presentan determinantes iguales a $1$, $-1$ y $0$, respectivamente, todos pertenecientes al conjunto:

$$
\{-1,0,1\}
$$

Estos resultados ilustran la propiedad de **total unimodularidad** de la matriz de incidencia nodo-arco. La propiedad no se demuestra únicamente mediante estas tres submatrices, sino que se sustenta teóricamente en que la matriz corresponde a la incidencia de una red dirigida.

La garantía de integralidad no proviene simplemente del solver. Se debe a la combinación de:

1. una matriz de incidencia totalmente unimodular;
2. un vector del lado derecho entero, con valores $+1$ y $-1$;
3. cotas enteras para las variables, entre 0 y 1.

Por ello, aunque el modelo permitió variables continuas entre 0 y 1, fue posible obtener una solución óptima 0-1 sin imponer explícitamente variables binarias.

# Restricción adicional C1-C2 ≤ 77

###
Hasta este punto se resolvió el problema de asignación como una red pura, obteniendo un óptimo de **236 minutos** y una solución naturalmente 0-1.

Ahora se incorpora la restricción adicional contenida en `restriccion_adicional.csv`.

Las celdas $C_1$ y $C_2$ comparten un único brazo de carga, por lo que la suma de los tiempos correspondientes a las órdenes asignadas a ambas celdas no puede superar el tope indicado en el archivo de datos.

Matemáticamente:

$$
\sum_{i\in O} t_{i,C_1}x_{i,C_1}
+
\sum_{i\in O} t_{i,C_2}x_{i,C_2}
\leq L
$$

donde $L$ es el tope de tiempo leído desde `restriccion_adicional.csv`.

Es importante notar que el valor del tope **no se escribe manualmente dentro del modelo**, sino que se obtiene directamente desde el archivo CSV entregado.

In [34]:
# Incorporación de la restricción adicional C1-C2

# Copiamos el modelo base para conservar intacta la solución original
m_relajado = m.clone()

# Leemos desde el CSV las celdas afectadas y el tope
celdas_brazo = restriccion.loc[0, "celdas"].split("+")
tope = float(restriccion.loc[0, "tope_minutos"])

print("Celdas que comparten el brazo:", celdas_brazo)
print("Tope de tiempo:", tope, "minutos")

# Restricción adicional:
# suma de los tiempos asignados a C1 y C2 <= tope leído desde el CSV
m_relajado.restriccion_brazo = pyo.Constraint(
    expr=sum(
        A[(o, c)][0] * m_relajado.x[o, c]
        for o in ordenes
        for c in celdas_brazo
    ) <= tope
)

Celdas que comparten el brazo: ['C1', 'C2']
Tope de tiempo: 77.0 minutos


### Relajación lineal con la restricción adicional

Se resuelve nuevamente el modelo incorporando la restricción del brazo compartido por $C_1$ y $C_2$, manteniendo las variables continuas:

$$
0 \leq x_{ij} \leq 1
$$

El objetivo es observar si la nueva restricción conserva o no la integralidad natural del problema base.

In [35]:
# Resolver el modelo continuo con la restricción adicional
res_relajado = pyo.SolverFactory("appsi_highs").solve(m_relajado)

print("Condición de término:", res_relajado.solver.termination_condition)
print("Valor óptimo relajado:", pyo.value(m_relajado.obj), "minutos")

# Guardar los valores de las variables
flujos_relajados = {
    a: pyo.value(m_relajado.x[a])
    for a in m_relajado.A
}

print("\nVariables positivas:")
for a, v in sorted(flujos_relajados.items()):
    if v > 1e-6:
        print(f"{a[0]:>2} -> {a[1]:<2}   {v:.4f}")

# Identificar variables fraccionarias
fraccionarias = {
    a: v for a, v in flujos_relajados.items()
    if v > 1e-6 and abs(v - round(v)) > 1e-6
}

print("\nVariables fraccionarias:")
for a, v in sorted(fraccionarias.items()):
    print(f"{a[0]:>2} -> {a[1]:<2}   {v:.4f}")

Condición de término: optimal
Valor óptimo relajado: 238.0 minutos

Variables positivas:
O1 -> C4   1.0000
O2 -> C2   0.6667
O2 -> C3   0.3333
O3 -> C1   1.0000
O4 -> C2   0.3333
O4 -> C5   0.6667
O5 -> C3   0.6667
O5 -> C5   0.3333
O6 -> C6   1.0000

Variables fraccionarias:
O2 -> C2   0.6667
O2 -> C3   0.3333
O4 -> C2   0.3333
O4 -> C5   0.6667
O5 -> C3   0.6667
O5 -> C5   0.3333


In [36]:
# Verificación de la restricción del brazo compartido

uso_brazo = sum(
    A[(o, c)][0] * pyo.value(m_relajado.x[o, c])
    for o in ordenes
    for c in celdas_brazo
)

print("Uso total del brazo C1-C2:", uso_brazo, "minutos")
print("Tope permitido:", tope, "minutos")
print("Restricción satisfecha:", uso_brazo <= tope + 1e-6)

Uso total del brazo C1-C2: 77.0 minutos
Tope permitido: 77.0 minutos
Restricción satisfecha: True


### Interpretación de la solución relajada

Al incorporar la restricción adicional del brazo compartido por $C_1$ y $C_2$, el valor óptimo aumenta desde **236 minutos** hasta **238 minutos**.

A diferencia del modelo base, ahora aparecen seis variables fraccionarias:

- $x_{O2,C2}=0.6667$
- $x_{O2,C3}=0.3333$
- $x_{O4,C2}=0.3333$
- $x_{O4,C5}=0.6667$
- $x_{O5,C3}=0.6667$
- $x_{O5,C5}=0.3333$

Aunque estas asignaciones satisfacen matemáticamente la relajación lineal, implican dividir una orden entre distintas celdas, lo que no corresponde a la decisión operacional requerida.

La restricción adicional se encuentra activa en el óptimo continuo, ya que el uso total del brazo alcanza exactamente los 77 minutos permitidos. Por lo tanto, al incorporar esta nueva condición se pierde la garantía de integralidad natural del modelo de red original y aparecen soluciones fraccionarias. Esto motiva resolver posteriormente el problema imponiendo variables binarias.

###  Modelo binario y brecha de integralidad

La solución relajada obtenida anteriormente contiene asignaciones fraccionarias, las cuales no son operacionalmente realizables si cada orden debe asignarse completamente a una sola celda.

Por esta razón, se resuelve nuevamente el mismo modelo, manteniendo la restricción adicional del brazo compartido por $C_1$ y $C_2$, pero imponiendo ahora:

$$
x_{ij}\in\{0,1\}
$$

De esta forma, cada variable puede tomar únicamente los valores 0 o 1.

In [37]:
# Modelo binario con la restricción adicional

# Copiamos el modelo relajado
m_binario = m_relajado.clone()

# El modelo binario no requiere importar valores duales
if hasattr(m_binario, "dual"):
    m_binario.del_component(m_binario.dual)

# Cambiamos el dominio de las variables a binarias
for a in m_binario.A:
    m_binario.x[a].domain = pyo.Binary

# Resolver
res_binario = pyo.SolverFactory("appsi_highs").solve(m_binario)

print("Condición de término:",
      res_binario.solver.termination_condition)

print("Óptimo entero:",
      pyo.value(m_binario.obj),
      "minutos")

print("\nAsignaciones óptimas:")

for a in sorted(m_binario.A):
    v = pyo.value(m_binario.x[a])

    if v > 0.5:
        print(f"{a[0]:>2} -> {a[1]:<2}   {v:.0f}")

Condición de término: optimal
Óptimo entero: 239.0 minutos

Asignaciones óptimas:
O1 -> C4   1
O2 -> C2   1
O3 -> C1   1
O4 -> C5   1
O5 -> C3   1
O6 -> C6   1


In [38]:
# Brecha entre la relajación lineal y el modelo entero

opt_relajado = pyo.value(m_relajado.obj)
opt_entero = pyo.value(m_binario.obj)

brecha_abs = opt_entero - opt_relajado
brecha_pct = 100 * brecha_abs / opt_entero

print("Óptimo relajación lineal:", opt_relajado, "minutos")
print("Óptimo entero:", opt_entero, "minutos")
print("Brecha absoluta:", brecha_abs, "minutos")
print("Brecha porcentual:", round(brecha_pct, 4), "%")

Óptimo relajación lineal: 238.0 minutos
Óptimo entero: 239.0 minutos
Brecha absoluta: 1.0 minutos
Brecha porcentual: 0.4184 %


### Interpretación

La relajación lineal entrega un valor óptimo de:

$$
z_{LP}=238\text{ minutos}
$$

Sin embargo, esta solución contiene asignaciones fraccionarias.

Al exigir que las variables sean binarias, el mejor valor factible es:

$$
z_{IP}=239\text{ minutos}
$$

La brecha absoluta de integralidad es:

$$
239-238=1\text{ minuto}
$$

y la brecha porcentual respecto del óptimo entero es:

$$
\frac{239-238}{239}\times100
\approx0.418\%
$$

Por lo tanto, exigir asignaciones completas 0-1 aumenta el tiempo óptimo en **1 minuto** respecto de la relajación lineal.

La relajación de 238 minutos funciona como una cota inferior para el problema entero, mientras que 239 minutos corresponde al mejor plan operacionalmente realizable bajo la restricción del brazo compartido.

###  Comparación de los tres escenarios

Se comparan los resultados obtenidos en las tres etapas del problema:

1. modelo base de asignación;
2. modelo continuo con la restricción adicional del brazo compartido;
3. modelo binario con la misma restricción adicional.

La comparación permite separar el efecto producido por la nueva restricción operacional del efecto producido por exigir asignaciones enteras.

In [39]:
# Comparación de los tres escenarios

opt_base = pyo.value(m.obj)
opt_relajado = pyo.value(m_relajado.obj)
opt_entero = pyo.value(m_binario.obj)

comparacion = pd.DataFrame({
    "Modelo": [
        "Base continuo",
        "Con restricción C1-C2, continuo",
        "Con restricción C1-C2, binario"
    ],
    "Tiempo óptimo (min)": [
        opt_base,
        opt_relajado,
        opt_entero
    ]
})

display(comparacion)

print("Efecto de incorporar la restricción:",
      opt_relajado - opt_base, "minutos")

print("Efecto de exigir integralidad:",
      opt_entero - opt_relajado, "minutos")

print("Diferencia total respecto del modelo base:",
      opt_entero - opt_base, "minutos")

,Modelo,Tiempo óptimo (min)
0,Base continuo,236.0
1,"Con restricción C1-C2, continuo",238.0
2,"Con restricción C1-C2, binario",239.0


Efecto de incorporar la restricción: 2.0 minutos
Efecto de exigir integralidad: 1.0 minutos
Diferencia total respecto del modelo base: 3.0 minutos


### Interpretación comparativa

Los tres modelos permiten distinguir el efecto de cada modificación realizada sobre el problema original.

El modelo base presenta un óptimo de **236 minutos**. Al incorporar la restricción operacional del brazo compartido por $C_1$ y $C_2$, manteniendo las variables continuas, el óptimo aumenta a **238 minutos**. Por tanto, el efecto de esta nueva restricción es:

$$
238-236=2\text{ minutos}
$$

Posteriormente, al exigir variables binarias, el óptimo aumenta desde 238 hasta **239 minutos**:

$$
239-238=1\text{ minuto}
$$

Este minuto adicional corresponde al efecto de exigir una solución entera y operacionalmente realizable.

En consecuencia, el paso desde el modelo base hasta el modelo operacional final produce una diferencia total de:

$$
239-236=3\text{ minutos}
$$

que puede descomponerse en **2 minutos asociados a la restricción del brazo compartido** y **1 minuto asociado a la exigencia de integralidad**.

### Pérdida de integralidad y necesidad de Branch-and-Bound

En el problema base de asignación, la matriz de restricciones corresponde a una matriz de incidencia de una red bipartita dirigida.

Esta matriz es **totalmente unimodular**, y como el vector del lado derecho es entero, la relajación lineal entrega soluciones enteras de manera natural, aun cuando las variables se declaren continuas:

$$
0 \leq x_{ij} \leq 1
$$

Por esta razón, en el modelo base se obtuvo una solución 0-1 sin imponer explícitamente variables binarias.

Al incorporar la restricción adicional del brazo compartido por $C_1$ y $C_2$:

$$
\sum_{i\in O} t_{i,C_1}x_{i,C_1}
+
\sum_{i\in O} t_{i,C_2}x_{i,C_2}
\leq L
$$

se agrega una nueva fila a la matriz de restricciones cuyos coeficientes corresponden a los tiempos de procesamiento $t_{ij}$.

Estos coeficientes ya no tienen la estructura propia de una matriz de incidencia, formada por valores $0$, $+1$ y $-1$.

Como consecuencia, se pierde la garantía de integralidad asociada a la estructura de red totalmente unimodular y, por lo tanto, también se pierde la garantía de que la relajación lineal produzca una solución entera.

Esto se observa directamente en los resultados obtenidos:

$$
z_{LP}=238\text{ minutos}
$$

con variables fraccionarias, mientras que al imponer:

$$
x_{ij}\in\{0,1\}
$$

se obtiene:

$$
z_{IP}=239\text{ minutos}
$$

Por lo tanto, el problema deja de poder resolverse únicamente como una red lineal pura y pasa a requerir técnicas de programación entera.

Una de estas técnicas es **Branch-and-Bound (ramificación y acotamiento)**.

El algoritmo parte resolviendo la relajación lineal. Si aparecen variables fraccionarias, selecciona una de ellas y divide el problema en ramas. Por ejemplo, si una variable presenta:

$$
x_{ij}=0.6667
$$

se generan dos subproblemas:

$$
x_{ij}=0
$$

y

$$
x_{ij}=1
$$

Cada rama se vuelve a resolver mediante programación lineal y entrega una cota para el mejor valor que podría alcanzarse en esa parte del espacio de soluciones.

Las ramas que no pueden mejorar la mejor solución entera encontrada son descartadas. Este proceso continúa hasta encontrar y demostrar el óptimo entero.

En nuestro caso, la relajación lineal de 238 minutos actúa como una cota inferior, mientras que la solución binaria de 239 minutos corresponde al mejor valor entero factible.

Verificación independiente del óptimo

In [40]:
from itertools import permutations

# Verificación independiente por enumeración exhaustiva.
# No se utiliza Pyomo ni HiGHS.

mejor_base = None
mejor_restringido = None
for perm in permutations(celdas):

    asignacion = dict(zip(ordenes, perm))

    # Tiempo total de esta asignación
    total = sum(
        A[(o, asignacion[o])][0]
        for o in ordenes
    )

    # Mejor solución del problema base
    if mejor_base is None or total < mejor_base[0]:
        mejor_base = (total, asignacion.copy())

    # Uso del brazo compartido C1-C2
    uso_brazo_enum = sum(
        A[(o, asignacion[o])][0]
        for o in ordenes
        if asignacion[o] in celdas_brazo
    )

    # Mejor solución que respeta la restricción adicional
    if uso_brazo_enum <= tope + 1e-6:
        if mejor_restringido is None or total < mejor_restringido[0]:
            mejor_restringido = (
                total,
                asignacion.copy(),
                uso_brazo_enum
            )

print("Número de asignaciones evaluadas:", 720)

print("\nÓptimo base por enumeración:")
print("Tiempo:", mejor_base[0], "minutos")
print("Asignación:", mejor_base[1])

print("\nÓptimo entero con restricción por enumeración:")
print("Tiempo:", mejor_restringido[0], "minutos")
print("Asignación:", mejor_restringido[1])
print("Uso brazo C1-C2:", mejor_restringido[2], "minutos")

Número de asignaciones evaluadas: 720

Óptimo base por enumeración:
Tiempo: 236 minutos
Asignación: {'O1': 'C4', 'O2': 'C3', 'O3': 'C1', 'O4': 'C2', 'O5': 'C5', 'O6': 'C6'}

Óptimo entero con restricción por enumeración:
Tiempo: 239 minutos
Asignación: {'O1': 'C4', 'O2': 'C2', 'O3': 'C1', 'O4': 'C5', 'O5': 'C3', 'O6': 'C6'}
Uso brazo C1-C2: 76 minutos


### Interpretación de la verificación independiente

La enumeración exhaustiva evaluó las $6!=720$ asignaciones uno-a-uno posibles entre las seis órdenes y las seis celdas, sin utilizar Pyomo ni el solver HiGHS.

Para el problema base, la enumeración confirmó un óptimo de:

$$
236\text{ minutos}
$$

coincidente con el resultado obtenido mediante programación lineal.

Al incorporar la restricción adicional del brazo compartido por $C_1$ y $C_2$, la enumeración confirmó que el mejor plan entero factible tiene un tiempo total de:

$$
239\text{ minutos}
$$

y utiliza:

$$
76\text{ minutos}
$$

de los 77 minutos disponibles para el brazo compartido, dejando una holgura de 1 minuto.

Por lo tanto, los óptimos enteros obtenidos mediante Pyomo y HiGHS quedan verificados mediante una segunda vía independiente.

## Paso 7. Valores duales

Informe cada dual con su **unidad** y declare la **convención de signos** que usa. Un dual sin
unidad no es interpretable y el informe pierde puntaje.


In [41]:
for n in N:
    print(f"{n:<10} {m.dual[m.bal[n]]:>14,.2f} min/unidad de orden")

# TODO: interprete. ¿Qué significa el dual de un nodo de demanda? ¿Y el de uno con holgura?


O1                  35.00 min/unidad de orden
O2                  34.00 min/unidad de orden
O3                  29.00 min/unidad de orden
O4                  37.00 min/unidad de orden
O5                  36.00 min/unidad de orden
O6                  38.00 min/unidad de orden
C1                  -7.00 min/unidad de orden
C2                  -6.00 min/unidad de orden
C3                 -10.00 min/unidad de orden
C4                  -3.00 min/unidad de orden
C5                  -1.00 min/unidad de orden
C6                  -0.00 min/unidad de orden


### Interpretación de los valores duales

Se utiliza la siguiente convención de balance:

$$
\text{salidas}-\text{entradas}=q_i
$$

donde los nodos de órdenes tienen $q_i=+1$ y los nodos de celdas tienen $q_j=-1$.

El valor dual $\pi_i$ asociado a la restricción de balance de un nodo representa la variación marginal del valor óptimo cuando el lado derecho $q_i$ cambia en una unidad, mientras la base óptima permanezca válida.

Como la función objetivo está expresada en **minutos** y $q_i$ en **unidades de orden**, la unidad de los valores duales es:

$$
\frac{\text{minutos}}{\text{unidad de orden}}
$$

#### Dual de un nodo de demanda

Para una celda robotizada se tiene:

$$
q_j=-1
$$

por lo que corresponde a un nodo de demanda.

Bajo la convención utilizada, aumentar $q_j$ significa hacerlo menos negativo, es decir, reducir la demanda. Por el contrario, aumentar la demanda en una unidad implica:

$$
\Delta q_j=-1
$$

Por ejemplo, para $C_1$ se obtuvo:

$$
\pi_{C_1}=-7
\frac{\text{minutos}}{\text{unidad de orden}}
$$

Por lo tanto, considerando el efecto marginal de ese balance:

$$
\Delta z
\approx
\pi_{C_1}\Delta q_{C_1}
=
(-7)(-1)
=
+7\text{ minutos}
$$

Sin embargo, como la red debe permanecer balanceada:

$$
\sum_{i\in N}q_i=0
$$

un cambio en la demanda de un nodo debe compensarse con una modificación correspondiente en otro nodo. Por esta razón, la interpretación operacional más adecuada se realiza mediante **diferencias entre potenciales nodales**, y no mediante un dual aislado.

#### Nodo con holgura

En esta instancia las restricciones de conservación de flujo son **igualdades**, por lo que los nodos de balance no presentan holgura.

En términos generales, si una restricción de desigualdad tiene holgura positiva en el óptimo, dicha restricción no está activa y su valor dual es cero. En ese caso, una pequeña relajación adicional de esa restricción no produce una mejora marginal del valor óptimo.

## Paso 8. Análisis de sensibilidad

La forma más segura y más transparente de obtener el valor de una capacidad es **volver a
resolver** con esa capacidad modificada, y comparar. Es lo que hace la función siguiente.


In [42]:
def resolver(A_mod, q_mod=None):
    """Resuelve una variante del modelo y devuelve (valor óptimo, flujos)."""
    q2 = q_mod if q_mod is not None else q
    mm = pyo.ConcreteModel()
    mm.A = pyo.Set(initialize=list(A_mod), dimen=2)
    mm.N = pyo.Set(initialize=list(q2))
    mm.x = pyo.Var(mm.A, domain=pyo.NonNegativeReals,
                   bounds=lambda mm, i, j: (A_mod[(i, j)][1], A_mod[(i, j)][2]))
    mm.obj = pyo.Objective(expr=sum(A_mod[a][0] * mm.x[a] for a in mm.A), sense=pyo.minimize)
    def bal(mm, n):
        sale  = sum(mm.x[i, j] for (i, j) in mm.A if i == n)
        entra = sum(mm.x[i, j] for (i, j) in mm.A if j == n)
        return sale - entra == q2[n]
    mm.bal = pyo.Constraint(mm.N, rule=bal)
    r = pyo.SolverFactory("appsi_highs").solve(mm)
    if str(r.solver.termination_condition) != "optimal":
        return None, None
    return pyo.value(mm.obj), {a: pyo.value(mm.x[a]) for a in mm.A}


# TODO: use resolver() para responder las preguntas de sensibilidad de su instancia.
# Ejemplo de uso: aumentar en una unidad la capacidad de un arco y medir el ahorro.


In [43]:
# Sensibilidad de una perturbación balanceada O1 - C4

valor_base = pyo.value(m.obj)

pi_O1 = m.dual[m.bal["O1"]]
pi_C4 = m.dual[m.bal["C4"]]

print("Valor óptimo base:", valor_base)
print("Diferencia dual O1-C4:", pi_O1 - pi_C4,
      "min/unidad de orden")

for delta in [-0.25, -0.50, -0.75, -1.00]:
    q_mod = q.copy()

    # Se reduce en delta la oferta de O1
    # y simultáneamente la demanda de C4,
    # manteniendo la red balanceada.
    q_mod["O1"] = q["O1"] + delta
    q_mod["C4"] = q["C4"] - delta

    valor_nuevo, _ = resolver(A, q_mod)

    prediccion_dual = valor_base + (pi_O1 - pi_C4) * delta

    print(
        f"delta={delta:5.2f} | "
        f"óptimo={valor_nuevo:7.2f} | "
        f"predicción dual={prediccion_dual:7.2f}"
    )

Valor óptimo base: 236.0
Diferencia dual O1-C4: 38.0 min/unidad de orden
delta=-0.25 | óptimo= 226.50 | predicción dual= 226.50
delta=-0.50 | óptimo= 217.00 | predicción dual= 217.00
delta=-0.75 | óptimo= 207.50 | predicción dual= 207.50
delta=-1.00 | óptimo= 198.00 | predicción dual= 198.00


In [44]:
# Comprobación del límite superior del rango de validez

for delta in [0.25, 0.50]:
    q_mod = q.copy()

    q_mod["O1"] = q["O1"] + delta
    q_mod["C4"] = q["C4"] - delta

    valor_nuevo, _ = resolver(A, q_mod)

    prediccion_dual = valor_base + (pi_O1 - pi_C4) * delta

    print(
        f"delta={delta:5.2f} | "
        f"óptimo={valor_nuevo:7.2f} | "
        f"predicción dual={prediccion_dual:7.2f}"
    )

delta= 0.25 | óptimo= 246.75 | predicción dual= 245.50
delta= 0.50 | óptimo= 257.50 | predicción dual= 255.00


### Interpretación del análisis de sensibilidad

Para la perturbación balanceada entre los nodos $O_1$ y $C_4$, la diferencia de potenciales duales es:

$$
\pi_{O_1}-\pi_{C_4}
=
35-(-3)
=
38
$$

por lo que el valor marginal relevante es:

$$
38\,
\frac{\text{minutos}}{\text{unidad de orden}}
$$

Para perturbaciones negativas se obtuvo:

| $\delta$ | Óptimo real | Predicción dual |
|---:|---:|---:|
| -0.25 | 226.50 | 226.50 |
| -0.50 | 217.00 | 217.00 |
| -0.75 | 207.50 | 207.50 |
| -1.00 | 198.00 | 198.00 |

En todos los valores evaluados, el cambio del valor óptimo coincide exactamente con la predicción:

$$
z(\delta)=236+38\delta
$$

Por lo tanto, los resultados obtenidos confirman que, para la perturbación balanceada analizada, la relación marginal asociada a los duales se mantiene válida en el intervalo considerado:

$$
-1\leq\delta\leq0
$$

El límite inferior $\delta=-1$ corresponde a reducir completamente la unidad de flujo asociada al arco $O_1\rightarrow C_4$.

En cambio, para perturbaciones positivas se obtuvo:

| $\delta$ | Óptimo real | Predicción dual |
|---:|---:|---:|
| 0.25 | 246.75 | 245.50 |
| 0.50 | 257.50 | 255.00 |

En estos casos el óptimo real deja de coincidir con la predicción dual. Esto ocurre porque el arco:

$$
O_1\rightarrow C_4
$$

ya se encontraba en su cota superior:

$$
x_{O_1,C_4}=1
$$

por lo que una perturbación positiva obliga al modelo a modificar la estructura de la solución óptima.

En consecuencia, los valores duales deben interpretarse como **tasas marginales locales**, válidas mientras se mantenga la estructura óptima asociada a la solución base. Cuando dicha estructura cambia, el valor dual original deja de predecir exactamente el nuevo valor óptimo y es necesario reoptimizar el modelo.

## Paso 9. Respuestas y limitaciones

Responda aquí, en prosa, las preguntas de su instancia. Cierre con la limitación del modelo que se
pide en la sección 4 del enunciado: un supuesto que el modelo hace, que la realidad no cumple, y en
qué dirección sesga la conclusión.

---

**Recordatorio de entrega:** repositorio con `README.md`, `datos/`, `modelo.py`,
`resultados/` y este cuaderno. Domingo 13 de septiembre, 23:59.


## Paso 9. Respuestas y limitaciones

### Pregunta 1

El problema se formuló como una red bipartita de flujo a costo mínimo, con seis nodos de órdenes y seis nodos de celdas. Cada orden posee flujo exógeno $+1$ y cada celda flujo exógeno $-1$, por lo que la red se encuentra balanceada. Cada arco representa una posible asignación orden-celda y su costo corresponde al tiempo de procesamiento.

### Pregunta 2

Al resolver el modelo base como programa lineal, manteniendo las variables continuas entre 0 y 1, se obtuvo un tiempo óptimo total de:

$$
236\text{ minutos}
$$

con la asignación:

$$
O_1\rightarrow C_4,\quad
O_2\rightarrow C_3,\quad
O_3\rightarrow C_1,
$$

$$
O_4\rightarrow C_2,\quad
O_5\rightarrow C_5,\quad
O_6\rightarrow C_6.
$$

Aunque no se impuso integralidad, todas las variables de la solución resultaron 0-1.

### Pregunta 3

La solución resultó entera debido a que la matriz de incidencia nodo-arco del problema de red es **totalmente unimodular** y el vector del lado derecho es entero.

Esta propiedad fue ilustrada mediante tres submatrices cuadradas cuyos determinantes fueron:

$$
1,\quad -1,\quad 0.
$$

### Pregunta 4

Al incorporar la restricción adicional asociada al brazo de carga compartido por $C_1$ y $C_2$, el óptimo continuo aumentó a:

$$
238\text{ minutos}.
$$

La solución dejó de ser entera y aparecieron las siguientes variables fraccionarias:

$$
x_{O_2,C_2}=0.6667,\qquad
x_{O_2,C_3}=0.3333,
$$

$$
x_{O_4,C_2}=0.3333,\qquad
x_{O_4,C_5}=0.6667,
$$

$$
x_{O_5,C_3}=0.6667,\qquad
x_{O_5,C_5}=0.3333.
$$

Además, el uso del brazo compartido alcanzó exactamente el límite de:

$$
77\text{ minutos}.
$$

### Pregunta 5

Al exigir variables binarias, el óptimo aumentó a:

$$
239\text{ minutos}.
$$

Por lo tanto, la brecha de integralidad respecto de la relajación lineal fue de:

$$
239-238=1\text{ minuto},
$$

equivalente aproximadamente a:

$$
0.418\%.
$$

### Pregunta 6

La restricción adicional incorpora coeficientes de tiempo que rompen la estructura pura de una matriz de incidencia de red. Por ello se pierde la garantía de integralidad asociada a la total unimodularidad y pueden aparecer soluciones fraccionarias.

Para recuperar decisiones 0-1 se requiere programación entera. Métodos como **Branch-and-Bound** resuelven relajaciones lineales, ramifican sobre variables fraccionarias y utilizan las cotas obtenidas para descartar ramas que no pueden mejorar la mejor solución entera conocida.

### Limitación del modelo

El modelo supone que los tiempos de procesamiento incluidos en la matriz representan completamente el tiempo real de cada asignación.

En una operación industrial pueden existir tiempos adicionales de preparación, cambio de herramientas, desplazamiento, espera o detención que no están considerados.

Como estos tiempos omitidos son no negativos, el modelo tiende a **subestimar el tiempo operacional real** y, por lo tanto, puede **sobreestimar la eficiencia del plan de asignación obtenido**.

In [45]:
from pathlib import Path

PROYECTO = Path("/content/taller_industria_4")

(PROYECTO / "datos").mkdir(parents=True, exist_ok=True)
(PROYECTO / "resultados").mkdir(parents=True, exist_ok=True)

print("Carpetas creadas.")

Carpetas creadas.


In [46]:
import shutil
from pathlib import Path

origen = Path("/content/datos")
destino = Path("/content/taller_industria_4/datos")

for archivo in origen.glob("*.csv"):
    shutil.copy2(archivo, destino / archivo.name)

print("CSV copiados correctamente.")

CSV copiados correctamente.


In [49]:
# Guardar la solución óptima entera

solucion = []

for a in sorted(m_binario.A):
    valor = pyo.value(m_binario.x[a])

    if valor > 0.5:
        orden, celda = a
        solucion.append({
            "orden": orden,
            "celda": celda,
            "x": valor,
            "tiempo_min": A[a][0]
        })

df_solucion = pd.DataFrame(solucion)

ruta_solucion = PROYECTO / "resultados" / "solucion.csv"
df_solucion.to_csv(ruta_solucion, index=False)

display(df_solucion)

print("Tiempo total:", df_solucion["tiempo_min"].sum(), "minutos")
print("Número de asignaciones:", df_solucion["x"].sum())

,orden,celda,x,tiempo_min
0,O1,C4,1.0,38
1,O2,C2,1.0,40
2,O3,C1,1.0,36
3,O4,C5,1.0,41
4,O5,C3,1.0,46
5,O6,C6,1.0,38


Tiempo total: 239 minutos
Número de asignaciones: 6.0


In [51]:
# Guardar valores duales del modelo lineal base

duales = []

for n in N:
    duales.append({
        "nodo": n,
        "q": q[n],
        "dual": m.dual[m.bal[n]],
        "unidad": "min/unidad de orden"
    })

df_duales = pd.DataFrame(duales)

ruta_duales = PROYECTO / "resultados" / "duales.csv"
df_duales.to_csv(ruta_duales, index=False)

display(df_duales)


print("Número de duales guardados:", len(df_duales))

,nodo,q,dual,unidad
0,O1,1,35.0,min/unidad de orden
1,O2,1,34.0,min/unidad de orden
2,O3,1,29.0,min/unidad de orden
3,O4,1,37.0,min/unidad de orden
4,O5,1,36.0,min/unidad de orden
5,O6,1,38.0,min/unidad de orden
6,C1,-1,-7.0,min/unidad de orden
7,C2,-1,-6.0,min/unidad de orden
8,C3,-1,-10.0,min/unidad de orden
9,C4,-1,-3.0,min/unidad de orden


Número de duales guardados: 12


In [52]:
modelo_py = r'''
from pathlib import Path
import pandas as pd
import pyomo.environ as pyo

# --------------------------------------------------
# 1. Lectura de datos
# --------------------------------------------------

BASE = Path(__file__).resolve().parent
CARPETA_DATOS = BASE / "datos"

tiempos = pd.read_csv(
    CARPETA_DATOS / "tiempos.csv",
    sep=";"
)

restriccion = pd.read_csv(
    CARPETA_DATOS / "restriccion_adicional.csv",
    sep=";"
)

# --------------------------------------------------
# 2. Conjuntos y parámetros
# --------------------------------------------------

ordenes = tiempos["orden"].tolist()
celdas = [col for col in tiempos.columns if col != "orden"]

A = {}

for _, fila in tiempos.iterrows():
    orden = fila["orden"]

    for celda in celdas:
        A[(orden, celda)] = fila[celda]

# Restricción adicional leída desde archivo
celdas_brazo = restriccion.loc[0, "celdas"].split("+")
tope = float(restriccion.loc[0, "tope_minutos"])

# --------------------------------------------------
# 3. Modelo
# --------------------------------------------------

m = pyo.ConcreteModel()

m.ORDENES = pyo.Set(initialize=ordenes)
m.CELDAS = pyo.Set(initialize=celdas)

m.x = pyo.Var(
    m.ORDENES,
    m.CELDAS,
    domain=pyo.Binary
)

# Función objetivo
m.obj = pyo.Objective(
    expr=sum(
        A[(o, c)] * m.x[o, c]
        for o in ordenes
        for c in celdas
    ),
    sense=pyo.minimize
)

# Cada orden se asigna a una celda
m.asignacion_orden = pyo.Constraint(
    m.ORDENES,
    rule=lambda m, o:
        sum(m.x[o, c] for c in celdas) == 1
)

# Cada celda recibe una orden
m.asignacion_celda = pyo.Constraint(
    m.CELDAS,
    rule=lambda m, c:
        sum(m.x[o, c] for o in ordenes) == 1
)

# Restricción del brazo compartido
m.restriccion_brazo = pyo.Constraint(
    expr=sum(
        A[(o, c)] * m.x[o, c]
        for o in ordenes
        for c in celdas_brazo
    ) <= tope
)

# --------------------------------------------------
# 4. Resolución
# --------------------------------------------------

resultado = pyo.SolverFactory("appsi_highs").solve(m)

print(
    "Condición de término:",
    resultado.solver.termination_condition
)

print(
    "Tiempo óptimo:",
    pyo.value(m.obj),
    "minutos"
)

print("\nAsignaciones:")

for o in ordenes:
    for c in celdas:
        if pyo.value(m.x[o, c]) > 0.5:
            print(
                f"{o} -> {c} | "
                f"{A[(o, c)]} minutos"
            )
'''

ruta_modelo = PROYECTO / "modelo.py"

ruta_modelo.write_text(
    modelo_py,
    encoding="utf-8"
)

print("Archivo creado:", ruta_modelo)

Archivo creado: /content/taller_industria_4/modelo.py


In [53]:
!python /content/taller_industria_4/modelo.py

Condición de término: optimal
Tiempo óptimo: 239.0 minutos

Asignaciones:
O1 -> C4 | 38 minutos
O2 -> C2 | 40 minutos
O3 -> C1 | 36 minutos
O4 -> C5 | 41 minutos
O5 -> C3 | 46 minutos
O6 -> C6 | 38 minutos


In [56]:
readme = """
# Taller 1 - Formulación y resolución de un modelo en red

## Integrantes
- Pedro Limarí
- Diego Baltazar

## Instancia asignada
Instancia D - Industria 4.0

## Objetivo

Este repositorio contiene el desarrollo del Taller 1 de Investigación de Operaciones.

El problema corresponde a una asignación entre órdenes y celdas robotizadas, formulada inicialmente como una red bipartita de flujo a costo mínimo.

Posteriormente se incorpora una restricción operacional adicional asociada a un brazo de carga compartido por las celdas C1 y C2, analizando su efecto sobre la integralidad de la solución.

## Estructura del repositorio

- datos/
  - tiempos.csv
  - restriccion_adicional.csv

- resultados/
  - solucion.csv
  - duales.csv

- modelo.py
  - Modelo final reproducible desarrollado en Python y Pyomo.

- cuaderno.ipynb
  - Desarrollo completo del taller, incluyendo formulación, resolución, verificación, integralidad, duales y análisis de sensibilidad.

## Metodología

1. Formulación del problema como red de flujo a costo mínimo.
2. Resolución del modelo lineal con variables continuas.
3. Verificación de la integralidad de la solución base.
4. Análisis de la matriz de incidencia y total unimodularidad.
5. Incorporación de la restricción adicional del brazo compartido.
6. Comparación entre relajación lineal y modelo binario.
7. Verificación independiente mediante enumeración exhaustiva.
8. Análisis de valores duales y sensibilidad.

## Resultado principal

El modelo binario con la restricción adicional obtiene un tiempo óptimo total de 239 minutos.

## Ejecución

Desde la carpeta principal del proyecto ejecutar:

python modelo.py

## Software utilizado

- Python
- Pyomo
- HiGHS
- pandas
- Google Colab

## Reproducibilidad

Los parámetros numéricos del problema se leen desde los archivos CSV contenidos en la carpeta datos/. De esta forma, los datos se mantienen separados del código del modelo.
"""

ruta_readme = PROYECTO / "README.md"

ruta_readme.write_text(
    readme,
    encoding="utf-8"
)

print("README creado correctamente:")
print(ruta_readme)

README creado correctamente:
/content/taller_industria_4/README.md
